# Data Pulling

This notebook pulls 6 raw datasets from public APIs: Yahoo Finance (health-sector ETF prices), FRED (macroeconomic indicators), Google Trends (symptom search volume), CDC FluView (influenza surveillance), NCHS (leading causes of death), and CMS (Medicare provider utilization). All datasets are saved to  for downstream cleaning.

## Setup

Uncomment and run the cell below to install required packages in the current Python environment.

In [2]:
# import sys
# !{sys.executable} -m pip install fredapi pytrends pandas yfinance

## Data Acquisition

Each section below fetches one dataset from its source API. Datasets are stored in named variables and saved at the end of this notebook.

### Yahoo Finance — Health Sector ETFs

Downloads 14 years of daily adjusted closing prices for four health-sector ETFs. These will be resampled to weekly frequency in .

In [3]:
import yfinance as yf
import pandas as pd

tickers = ['XLV', 'XBI', 'PJP', 'KIE']
# XLV - Health Care Select Sector SPDR Fundabs
# XBI - SPDR S&P Biotech ETFabs
# PJP - Invesco Dynamic Pharmaceuticals ETF
# KIE - SPDR S&P Insurance ETF
yf_data = yf.download(tickers, start='2010-01-01', end='2024-01-01')


[*********************100%***********************]  4 of 4 completed


### FRED — Macroeconomic Indicators

Pulls three monthly macroeconomic series from the Federal Reserve Economic Data API: CPI (inflation), the unemployment rate, and the effective federal funds rate. These serve as macroeconomic controls in the analysis.

In [4]:
from fredapi import Fred
import os 

fred = Fred(api_key=os.getenv('FRED_API_KEY'))      
cpi = fred.get_series('CPIAUCSL', observation_start='2010-01-01')
# Consumer Price Index for All Urban Consumers

unemployment = fred.get_series('UNRATE', observation_start='2010-01-01')
interest_rate = fred.get_series('FEDFUNDS', observation_start='2010-01-01')


In [5]:
print(cpi.head(10))
print(cpi.shape)

2010-01-01    217.488
2010-02-01    217.281
2010-03-01    217.353
2010-04-01    217.403
2010-05-01    217.290
2010-06-01    217.199
2010-07-01    217.605
2010-08-01    217.923
2010-09-01    218.275
2010-10-01    219.035
dtype: float64
(197,)


The CPI series contains 197 monthly observations from 2010 through 2023, confirming the full study window is covered. All three FRED series share the same monthly cadence and will be forward-filled to weekly frequency during cleaning.

### Google Trends — Symptom Search Volume

Fetches weekly relative search interest (0–100 scale) for three respiratory symptom queries in the US. Because the Trends API limits a single request to a 5-year window, the full date range is split into four chunks and concatenated. A 30-second sleep between requests prevents rate-limiting.

In [6]:
from pytrends.request import TrendReq
import pandas as pd
import time

pytrends = TrendReq(hl='en-US', tz=360)
keywords = ['flu symptoms', 'fever', 'shortness of breath']

timeframes = [
    '2010-01-01 2013-12-31',
    '2014-01-01 2017-12-31',
    '2018-01-01 2021-12-31',
    '2022-01-01 2024-01-01'
]

chunks = []
for tf in timeframes:
    pytrends.build_payload(keywords, timeframe=tf, geo='US')
    time.sleep(30)  # wait 30 seconds between each request
    chunk = pytrends.interest_over_time()
    chunk = chunk.drop(columns=['isPartial'], errors='ignore')
    chunks.append(chunk)
    print(f"Done: {tf}")

trends_df = pd.concat(chunks)
trends_df = trends_df[~trends_df.index.duplicated(keep='first')]

print(trends_df.shape)
print(trends_df.head())

Done: 2010-01-01 2013-12-31
Done: 2014-01-01 2017-12-31
Done: 2018-01-01 2021-12-31
Done: 2022-01-01 2024-01-01
(732, 3)
            flu symptoms  fever  shortness of breath
date                                                
2009-12-27            20     44                    3
2010-01-03            16     44                    2
2010-01-10            14     46                    2
2010-01-17            13     46                    3
2010-01-24            11     50                    3


### CDC FluView — Influenza-Like Illness Surveillance

Queries the CMU Delphi Epidata REST API, which mirrors CDC FluView data with a clean programmatic interface. Returns weekly national-level ILI metrics including weighted ILI percentage (), raw ILI percentage (), patient counts, and age-group breakdowns.

In [7]:
import requests
import pandas as pd

# CMU Delphi Epidata API - built by Carnegie Mellon for epidemiological research
# It mirrors CDC FluView data but exposes it through a clean, reliable REST API
# epiweeks=200001-202401 means every flu week from 2000 week 1 to 2024 week 1
# regions=nat means national level data

url = "https://api.delphi.cmu.edu/epidata/fluview/"
params = {
    "regions": "nat",
    "epiweeks": "201001-202401"  # 2010 week 1 through 2024 week 1
}

resp = requests.get(url, params=params, timeout=30)
data = resp.json()

# The API returns a dict with 'result', 'message', and 'epidata' keys
# epidata is the list of weekly records
fluview_df = pd.DataFrame(data['epidata'])

print(fluview_df.shape)
print(fluview_df.head())

(731, 16)
  release_date region   issue  epiweek  lag  num_ili  num_patients  \
0   2013-12-31    nat  201352   201001  207    14299        721138   
1   2013-12-31    nat  201352   201002  206    14088        770895   
2   2013-12-31    nat  201352   201003  205    14757        766177   
3   2013-12-31    nat  201352   201004  204    15122        785580   
4   2013-12-31    nat  201352   201005  203    16037        767773   

   num_providers  num_age_0  num_age_1 num_age_2  num_age_3  num_age_4  \
0           1996       4998       3961      None       3333       1244   
1           2016       4877       4614      None       2793       1182   
2           2053       5399       5079      None       2693       1008   
3           2026       5333       5655      None       2560       1046   
4           1996       5816       6142      None       2581        948   

   num_age_5      wili       ili  
0        763  1.907118  1.982838  
1        622  1.867375  1.827486  
2        578  1.880

### NCHS — Leading Causes of Death

Retrieves annual death counts from the CDC WONDER dataset via the Socrata Open Data API. This is state-level, cause-level annual data; it will be aggregated to national totals and forward-filled to weekly frequency in cleaning. The  parameter overrides the default 1,000-row cap.

In [8]:
import requests
import pandas as pd

# data.cdc.gov uses the Socrata Open Data API (SODA)
# The resource ID 'bi63-dtpu' points to the NCHS Leading Causes of Death dataset
# .json at the end tells it to return JSON format
url = "https://data.cdc.gov/resource/bi63-dtpu.json"

# $limit=50000 tells the API to return up to 50,000 rows
# Without this it defaults to 1,000 rows which won't get you everything
params = {"$limit": 50000}

resp = requests.get(url, params=params, timeout=30)

# resp.json() parses the response into a list of dictionaries
# pd.DataFrame() converts that list into a dataframe, one row per record
wonder_df = pd.DataFrame(resp.json())

print(wonder_df.shape)
print(wonder_df.head())

(10868, 6)
   year                                    _113_cause_name      cause_name  \
0  2012  Nephritis, nephrotic syndrome and nephrosis (N...  Kidney disease   
1  2017  Nephritis, nephrotic syndrome and nephrosis (N...  Kidney disease   
2  2016  Nephritis, nephrotic syndrome and nephrosis (N...  Kidney disease   
3  2013  Nephritis, nephrotic syndrome and nephrosis (N...  Kidney disease   
4  2000  Intentional self-harm (suicide) (*U03,X60-X84,...         Suicide   

                  state deaths aadr  
0               Vermont     21  2.6  
1               Vermont     29  3.3  
2               Vermont     30  3.7  
3               Vermont     30  3.8  
4  District of Columbia     23  3.8  


### CMS — Medicare Inpatient Provider Data

Paginates through the CMS inpatient prospective payment dataset. Because the API caps responses at 1,000 rows per request, the loop increments an offset until a partial page signals the end of data. A  guard prevents runaway requests.

In [9]:
import requests
import pandas as pd

url = "https://data.cms.gov/data-api/v1/dataset/690ddc6c-2767-4618-b277-420ffb2bf27c/data"

# We don't know how many rows exist so we keep fetching pages
# until a page returns less than 1000 rows, which means we hit the end
all_records = []
offset = 0
page_size = 1000
max_pages = 10  # safety limit to prevent infinite loops

for page in range(max_pages):
    resp = requests.get(url, params={"size": page_size, "offset": offset})
    batch = resp.json()
    all_records.extend(batch)
    
    # If we got less than a full page, we're done
    if len(batch) < page_size:
        break
    
    # Move to the next page
    offset += page_size

cms_df = pd.DataFrame(all_records)
print(cms_df.shape)
print(cms_df.head())

(10000, 15)
  Rndrng_Prvdr_CCN            Rndrng_Prvdr_Org_Name Rndrng_Prvdr_City  \
0           010001  Southeast Health Medical Center            Dothan   
1           010001  Southeast Health Medical Center            Dothan   
2           010001  Southeast Health Medical Center            Dothan   
3           010001  Southeast Health Medical Center            Dothan   
4           010001  Southeast Health Medical Center            Dothan   

          Rndrng_Prvdr_St Rndrng_Prvdr_State_FIPS Rndrng_Prvdr_Zip5  \
0  1108 Ross Clark Circle                      01             36301   
1  1108 Ross Clark Circle                      01             36301   
2  1108 Ross Clark Circle                      01             36301   
3  1108 Ross Clark Circle                      01             36301   
4  1108 Ross Clark Circle                      01             36301   

  Rndrng_Prvdr_State_Abrvtn Rndrng_Prvdr_RUCA  \
0                        AL                 2   
1                       

In [10]:
resp = requests.get(
    "https://data.cms.gov/data-api/v1/dataset/690ddc6c-2767-4618-b277-420ffb2bf27c/data",
    params={"size": 1000, "offset": 1000}
)
print(resp.status_code)
print(len(resp.json()))

200
1000


The 200 status code and a full 1,000-row second page confirm the pagination loop is working correctly and that more data follows the first page.

## Saving Raw Datasets

All six raw datasets are written to  as CSV files before any transformation. Saving raw data separately from cleaned data ensures the pulling step only needs to run once and that intermediate cleaning decisions can be revisited without re-hitting the APIs.

In [12]:
import os
raw_path = '../data/raw/'

# yfinance
yf_data['Close'].to_csv(os.path.join(raw_path, 'yfinance_raw.csv'))

# FRED
pd.DataFrame({'cpi': cpi, 'unemployment': unemployment, 'interest_rate': interest_rate}).to_csv(os.path.join(raw_path, 'fred_raw.csv'))

# Google Trends
trends_df.to_csv(os.path.join(raw_path, 'google_trends_raw.csv'))

# FluView
fluview_df.to_csv(os.path.join(raw_path, 'fluview_raw.csv'))

# NCHS
wonder_df.to_csv(os.path.join(raw_path, 'nchs_raw.csv'))

# CMS
cms_df.to_csv(os.path.join(raw_path, 'cms_raw.csv'))

print("All raw datasets saved.")

All raw datasets saved.
